# Generate 1,000 Diverse Alpha Factors

This notebook generates ~1,000 alpha factors using the Anthropic API,
with expanded themes and diversity prompts to explore more of the expression
space than the baseline's 216 factors.

**Themes (12 vs. baseline's 4):**
- Core: momentum, mean reversion, volume-price relationship, volatility
- Extended: liquidity, price pattern, volume dynamics, cross-signal interaction,
  tail risk, intraday structure, trend strength, composite signals

**Diversity controls:**
- Explicit prompt guidance to use less-common operators (ts_corr, power, log)
- Encouragement for deeper nesting (depth 3-4) and interaction terms
- Deduplication by exact expression string after generation
- Validation: AST parsing + operator/field whitelist

**Output:** `../results/evaluation/factors_1000.json`


In [ ]:
import os
import re
import json
import ast
import time
import anthropic

# Load API key from environment
ANTHROPIC_API_KEY = "Your_API_Key_Here"
if not ANTHROPIC_API_KEY:
    raise RuntimeError(
        "ANTHROPIC_API_KEY not set. Run: export ANTHROPIC_API_KEY=sk-ant-..."
    )
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
print("API client ready")


API client ready


In [6]:
OPERATORS = """
Time-series operators (look back over past d days for each stock):
  ts_mean(x, d)     — rolling mean over d days
  ts_std(x, d)      — rolling standard deviation over d days
  ts_corr(x, y, d)  — rolling correlation between x and y over d days
  ts_delta(x, d)    — difference: today's value minus the value d days ago
  ts_max(x, d)      — rolling maximum over d days
  ts_min(x, d)      — rolling minimum over d days

Cross-sectional operators (compare across all stocks on the same day):
  rank(x)           — rank each stock from 0 to 1 (percentile)
  zscore(x)         — standardize: subtract mean, divide by std

Math operators:
  sign(x)           — returns +1, -1, or 0
  log(x)            — natural log
  abs(x)            — absolute value
  power(x, n)       — x to the power of n

Arithmetic: +  -  *  /
"""

DATA_FIELDS = """
Available data fields (each is a matrix: rows = dates, columns = stocks):
  open    — opening price
  high    — daily high price
  low     — daily low price
  close   — closing price
  volume  — trading volume
  vwap    — volume-weighted average price
  returns — daily return: (today - yesterday) / yesterday
  adv20   — 20-day average daily volume
"""

EXAMPLES = [
    {"factor": "rank(ts_corr(close, volume, 10))",
     "description": "10-day price-volume correlation rank"},
    {"factor": "ts_delta(close, 5) / ts_std(close, 20)",
     "description": "5-day momentum / 20-day volatility"},
    {"factor": "rank(close / ts_mean(close, 20))",
     "description": "Price deviation from 20-day MA"},
    {"factor": "sign(ts_mean(volume, 10) - volume)",
     "description": "Volume below 10-day average"},
    {"factor": "rank(volume / adv20)",
     "description": "Relative volume vs 20-day avg"},
    {"factor": "rank(ts_corr(abs(returns), volume / adv20, 15))",
     "description": "Correlation of volatility surprise with relative volume"},
    {"factor": "-1 * rank(ts_delta(close, 10) / ts_std(close, 20)) * rank(volume / adv20)",
     "description": "Volatility-normalized reversal weighted by volume anomaly"},
    {"factor": "rank(power(ts_corr(vwap, volume, 10), 2)) * sign(ts_delta(close, 5))",
     "description": "Squared price-volume correlation modulated by momentum direction"},
]


def validate_factor(expr):
    ALLOWED_OPS = {
        "ts_mean", "ts_std", "ts_corr", "ts_delta", "ts_max", "ts_min",
        "rank", "zscore", "sign", "log", "abs", "power"
    }
    ALLOWED_FIELDS = {
        "open", "high", "low", "close", "volume", "vwap", "returns", "adv20"
    }
    try:
        tree = ast.parse(expr, mode="eval")
    except SyntaxError as e:
        return False, f"Syntax error: {e}"

    errors = []
    class Checker(ast.NodeVisitor):
        def visit_Call(self, node):
            if isinstance(node.func, ast.Name):
                if node.func.id not in ALLOWED_OPS:
                    errors.append(f"Unknown operator: {node.func.id}")
            self.generic_visit(node)
        def visit_Name(self, node):
            if node.id not in ALLOWED_FIELDS and node.id not in ALLOWED_OPS:
                errors.append(f"Unknown field: {node.id}")
        def visit_Attribute(self, node):
            errors.append("Attribute access not allowed")

    Checker().visit(tree)
    if errors:
        return False, "; ".join(errors)
    return True, ""

print(f"Defined {len(EXAMPLES)} examples, validator ready")


Defined 8 examples, validator ready


In [7]:
# 12 themes with specific guidance for diversity
THEMES = [
    {
        "name": "momentum",
        "prompt": "short-term and medium-term price momentum signals. "
                  "Use ts_delta, ts_mean of returns, and momentum ratios. "
                  "Include some factors that combine momentum with volume confirmation."
    },
    {
        "name": "mean reversion",
        "prompt": "mean reversion and contrarian signals. "
                  "Focus on price deviation from moving averages, "
                  "Bollinger-band style signals, and oversold/overbought indicators."
    },
    {
        "name": "volume-price divergence",
        "prompt": "divergence between price and volume trends. "
                  "Use ts_corr between price-based and volume-based quantities. "
                  "Focus on signals where volume disagrees with price direction."
    },
    {
        "name": "volatility regimes",
        "prompt": "volatility-based signals including volatility clustering, "
                  "volatility breakouts, and vol-of-vol. Use ts_std at multiple "
                  "timeframes and ratios between them. Include power() for higher moments."
    },
    {
        "name": "liquidity and flow",
        "prompt": "liquidity signals: volume relative to adv20, volume trends, "
                  "abnormal volume spikes. Use log(volume/adv20), ts_delta of volume, "
                  "and interaction between volume changes and price changes."
    },
    {
        "name": "intraday structure",
        "prompt": "intraday price patterns using open, high, low, close relationships. "
                  "Examples: upper/lower shadow ratios, body-to-range ratios, "
                  "open-to-close vs high-to-low, and candlestick-inspired features."
    },
    {
        "name": "trend strength and persistence",
        "prompt": "trend strength and persistence measures. Compare short vs long "
                  "moving averages, use ts_corr(close, time_proxy, d) style signals, "
                  "consecutive direction counters via sign and ts_mean of sign(returns)."
    },
    {
        "name": "tail risk and extreme moves",
        "prompt": "tail risk and extreme move signals. Use ts_max, ts_min of returns, "
                  "power(returns, 3) for skewness proxies, max drawdown proxies "
                  "like (close - ts_max(close, 20)) / ts_max(close, 20), "
                  "and abs(returns) / ts_std(returns, d) for standardized shocks."
    },
    {
        "name": "cross-signal interactions",
        "prompt": "interaction terms that multiply or divide two distinct signals. "
                  "Examples: momentum × volume anomaly, volatility × reversal, "
                  "correlation × trend. Each factor should combine TWO different "
                  "economic ideas into a single expression using * or /."
    },
    {
        "name": "multi-timeframe",
        "prompt": "multi-timeframe signals comparing the same metric at different "
                  "lookback windows. Examples: ts_mean(close, 5) / ts_mean(close, 20), "
                  "ts_std(returns, 5) / ts_std(returns, 20), "
                  "ts_delta(close, 3) - ts_delta(close, 10). "
                  "Use ratios and differences across 3/5/10/20-day windows."
    },
    {
        "name": "VWAP-based signals",
        "prompt": "signals centered on VWAP. VWAP deviation from close, "
                  "rolling VWAP trends, VWAP vs moving averages, "
                  "ts_corr(vwap, close, d) vs ts_corr(vwap, volume, d) divergence."
    },
    {
        "name": "composite and deeply nested",
        "prompt": "complex composite factors with 3-4 levels of nesting. "
                  "Each factor should use at least 3 different operators. "
                  "Example structure: rank(ts_corr(ts_delta(x, d1), ts_mean(y, d2), d3)). "
                  "Aim for creative combinations that capture non-obvious relationships."
    },
]

print(f"Defined {len(THEMES)} themes")
for t in THEMES:
    print(f"  - {t['name']}")


Defined 12 themes
  - momentum
  - mean reversion
  - volume-price divergence
  - volatility regimes
  - liquidity and flow
  - intraday structure
  - trend strength and persistence
  - tail risk and extreme moves
  - cross-signal interactions
  - multi-timeframe
  - VWAP-based signals
  - composite and deeply nested


In [8]:
SYSTEM_PROMPT = f"""You are a quantitative researcher designing Alpha factors for stock markets.

Available operators:
{OPERATORS}

Available data fields:
{DATA_FIELDS}

Rules:
- Output a JSON array only, no extra text
- Each item must have "factor" (the expression) and "description" (the economic intuition)
- Only use the operators and fields listed above
- Aim for DIVERSE expressions: vary nesting depth, operator combinations, and timeframes
- Prefer factors with 2-4 levels of operator nesting over simple single-operator wrappers
- Use a MIX of timeframes: 3, 5, 10, 15, 20 day windows
- Include interaction terms (multiplying/dividing two sub-signals) in at least 30% of factors
"""


def generate_batch(n_factors, theme_prompt, round_num=0):
    """Generate a batch of factors for a given theme."""
    examples_str = "\n".join(
        f'  {{"factor": "{e["factor"]}", "description": "{e["description"]}"}}' 
        for e in EXAMPLES
    )

    user_msg = f"""Here are example Alpha factors for reference:
[
{examples_str}
]

Theme: {theme_prompt}

Generate {n_factors} new, diverse Alpha factors for this theme.
- Make each factor structurally DIFFERENT (vary operators, nesting, timeframes)
- Avoid repeating the same pattern with only minor parameter changes
- Round {round_num + 1}: aim for {'simple to moderate complexity' if round_num == 0 else 'moderate to high complexity with deeper nesting'}
Output JSON array only."""

    try:
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=8096,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}]
        )
        raw = next(b.text for b in response.content if b.type == "text").strip()
        raw = re.sub(r"```(?:json)?|```", "", raw).strip()
        return json.loads(raw)
    except Exception as e:
        print(f"    Generation failed: {e}")
        return []


print("Generation function ready")


Generation function ready


In [9]:
# ── Main generation loop ──────────────────────────────────────────────────────
#
# Strategy: 2 rounds per theme, 15 factors per batch, 12 themes
# = 2 × 15 × 12 = 360 raw per round-set, ~720 total raw
# After dedup + validation, expect ~600-800 usable
# Combined with existing 216, should reach ~800-1000

CACHE_PATH = "../results/evaluation/factors_1000.json"
BATCH_SIZE = 15
N_ROUNDS = 2

# Load existing factors
existing = []
if os.path.exists(CACHE_PATH):
    with open(CACHE_PATH, "r") as f:
        existing = json.load(f)
    print(f"Loaded {len(existing)} existing factors from cache")

# Also load the baseline 216
baseline_path = "../results/evaluation/factors.json"
if os.path.exists(baseline_path) and len(existing) == 0:
    with open(baseline_path, "r") as f:
        existing = json.load(f)
    print(f"Seeded with {len(existing)} baseline factors")

all_factors = list(existing)
seen = {f["factor"] for f in existing}
n_generated = 0
n_rejected = 0
n_duplicate = 0

print(f"Starting with {len(all_factors)} factors, targeting ~1000")
print(f"Generating: {len(THEMES)} themes × {N_ROUNDS} rounds × {BATCH_SIZE} per batch\n")

for round_num in range(N_ROUNDS):
    print(f"═══ Round {round_num + 1}/{N_ROUNDS} ═══")
    for theme in THEMES:
        print(f"  Theme: {theme['name']}", end=" → ")
        batch = generate_batch(BATCH_SIZE, theme["prompt"], round_num)
        
        added = 0
        for f in batch:
            expr = f.get("factor", "").strip()
            if not expr:
                continue
            n_generated += 1
            
            if expr in seen:
                n_duplicate += 1
                continue
            
            ok, err = validate_factor(expr)
            if not ok:
                n_rejected += 1
                continue
            
            seen.add(expr)
            f["theme"] = theme["name"]
            f["round"] = round_num
            all_factors.append(f)
            added += 1
        
        print(f"{added} new factors (batch had {len(batch)})")
        time.sleep(0.3)  # rate limit courtesy
    
    # Save after each round
    with open(CACHE_PATH, "w") as f:
        json.dump(all_factors, f, indent=2)
    print(f"  Saved {len(all_factors)} factors to {CACHE_PATH}\n")

print(f"\n{'═'*60}")
print(f"Generation complete:")
print(f"  Total factors:     {len(all_factors)}")
print(f"  Raw generated:     {n_generated}")
print(f"  Duplicates:        {n_duplicate}")
print(f"  Rejected (invalid):{n_rejected}")
print(f"  New factors added: {len(all_factors) - len(existing)}")


Seeded with 216 baseline factors
Starting with 216 factors, targeting ~1000
Generating: 12 themes × 2 rounds × 15 per batch

═══ Round 1/2 ═══
  Theme: momentum → 15 new factors (batch had 15)
  Theme: mean reversion → 15 new factors (batch had 15)
  Theme: volume-price divergence → 15 new factors (batch had 15)
  Theme: volatility regimes → 13 new factors (batch had 15)
  Theme: liquidity and flow → 15 new factors (batch had 15)
  Theme: intraday structure → 15 new factors (batch had 15)
  Theme: trend strength and persistence → 15 new factors (batch had 15)
  Theme: tail risk and extreme moves → 15 new factors (batch had 15)
  Theme: cross-signal interactions → 15 new factors (batch had 15)
  Theme: multi-timeframe → 15 new factors (batch had 15)
  Theme: VWAP-based signals → 14 new factors (batch had 15)
  Theme: composite and deeply nested → 15 new factors (batch had 15)
  Saved 393 factors to ../results/evaluation/factors_1000.json

═══ Round 2/2 ═══
  Theme: momentum → 15 new fac

In [ ]:
# ── Diversity analysis ─────────────────────────────────────────────────────────

with open(CACHE_PATH, "r") as f:
    all_factors = json.load(f)

print(f"Total factors: {len(all_factors)}")

# Structural analysis
import collections

op_counts = collections.Counter()
depth_counts = collections.Counter()
top_level_counts = collections.Counter()

def max_depth(node, d=0):
    if isinstance(node, ast.Call):
        child_depths = [max_depth(c, d+1) for c in ast.iter_child_nodes(node)]
        return max(child_depths) if child_depths else d+1
    child_depths = [max_depth(c, d) for c in ast.iter_child_nodes(node)]
    return max(child_depths) if child_depths else d

for f in all_factors:
    expr = f["factor"]
    try:
        tree = ast.parse(expr, mode="eval").body
        
        # Nesting depth
        depth_counts[max_depth(tree)] += 1
        
        # Top-level pattern
        if isinstance(tree, ast.Call) and isinstance(tree.func, ast.Name):
            top_level_counts[tree.func.id] += 1
        elif isinstance(tree, ast.BinOp):
            top_level_counts["arithmetic"] += 1
        elif isinstance(tree, ast.UnaryOp):
            top_level_counts["negation"] += 1
        else:
            top_level_counts["other"] += 1
        
        # Operator usage
        def walk(node):
            if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
                op_counts[node.func.id] += 1
            for c in ast.iter_child_nodes(node): walk(c)
        walk(tree)
    except:
        pass

print(f"\nNesting depth distribution:")
for d in sorted(depth_counts):
    bar = "█" * (depth_counts[d] // 5)
    print(f"  depth {d}: {depth_counts[d]:>4}  {bar}")

print(f"\nTop-level operator:")
for op, c in top_level_counts.most_common():
    print(f"  {op:>15}: {c}")

print(f"\nOperator usage (total calls across all factors):")
for op, c in op_counts.most_common():
    print(f"  {op:>10}: {c}")

# Theme distribution (if tagged)
theme_counts = collections.Counter()
for f in all_factors:
    theme_counts[f.get("theme", "baseline")] += 1
print(f"\nFactors by theme:")
for theme, c in theme_counts.most_common():
    print(f"  {theme:>30}: {c}")


## Next steps

The generated factors are saved to `../results/evaluation/factors_1000.json`.

To evaluate them through the optimized pipeline:
1. Open your `opt6.ipynb` notebook
2. Change the factor loading cell to point to `factors_1000.json`:
   ```python
   with open("../results/evaluation/factors_1000.json", "r") as f:
       factors = json.load(f)
   ```
3. Run the pipeline — `compute_all_factors` should handle ~1000 factors in ~4s
4. `evaluate_all_factors` will take longer (~750s at current speed) — consider
   vectorizing it as a next optimization step

Analysis to consider:
- Score distribution: what % of factors have Sharpe > 1? ICIR > 0.1?
- Theme comparison: which themes produce the best factors on average?
- Diversity vs quality tradeoff: do deeply-nested factors outperform simple ones?
- Diminishing returns: plot factor quality vs generation rank to see if the
  LLM's creativity degrades over time
